# Silver - Gold | CineData Analytics

**Entregas desta camada**
1. **Star Schema** para o time de BI: 1 fato + 5 dimensões + 3 tabelas-ponte (*bridge*).
2. **`gold.gold_genai_movies_context`**: documento de texto por filme para o *Vector Search* (RAG) do time de IA.
3. **Desafio de Analytics**: 6 perguntas de negócio respondidas com `display()`.

```
                      dim_genres  <──  bridge_movie_genre  ──┐
                      dim_people  <──  bridge_movie_person  ──┼──>  dim_movies  <──  fact_movies_performance
                      dim_companies <─ bridge_movie_company ──┘          ▲
                                                                dim_reviews (1 linha por filme)
```

**Chaves substitutas (Surrogate Keys):** `BIGINT` gerado por **hash SHA-256** da chave natural (primeiros 60 bits).
São **determinísticas** — reprocessar o pipeline gera exatamente as mesmas chaves, sem quebrar relatórios do BI que já usam o modelo.

In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalog", "workspace", "1. Catálogo (vazio = padrão da sessão)")
dbutils.widgets.text("somente_lancados", "nao", "2. Fato somente com filmes 'Lançado'? (sim/nao)")

CATALOG = dbutils.widgets.get("catalog").strip()
SOMENTE_LANCADOS = dbutils.widgets.get("somente_lancados").strip().lower() in ("sim", "s", "true", "1", "yes")
if CATALOG:
    spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

def sk(*colunas, prefixo: str = ""):
    """
    Surrogate key BIGINT determinística: SHA-256 da chave natural -> 15 primeiros dígitos hex (60 bits) -> BIGINT.
    Probabilidade de colisão desprezível para o volume da base (~1e-7 com 1 milhão de registros).
    O 'prefixo' evita que tabelas diferentes gerem a mesma chave para o mesmo valor.
    """
    partes = [F.lit(prefixo)] + [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in colunas]
    return F.conv(F.substring(F.sha2(F.concat_ws("||", *partes), 256), 1, 15), 16, 10).cast("bigint")

def gravar_gold(df, tabela: str):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"✔ {tabela:<36} {spark.table(tabela).count():>10} linhas")

s_info   = spark.table("silver.tb_info_filmes")
s_fin    = spark.table("silver.tb_financeiro_filmes")
s_met    = spark.table("silver.tb_metricas_engajamento")
s_rev    = spark.table("silver.tb_avaliacoes_usuarios")
s_gen    = spark.table("silver.tb_generos")
s_pessoa = spark.table("silver.tb_pessoas_empresas")

## Entrega 1 — Modelagem Dimensional (Star Schema)

### 1.1 Dimensões

In [0]:
dim_movies = s_info.select(
    sk("id_filme", prefixo="movie").alias("sk_movie_id"),
    F.col("id_filme").cast("string").alias("id_filme"),                
    F.col("titulo").cast("string"),
    F.col("data_lancamento").cast("date"),
    F.col("ano_lancamento").cast("int"),
    F.col("duracao_minutos").cast("int"),
    F.col("idioma_original").cast("string"),
    F.col("status_filme").cast("string"),
    F.col("sinopse").cast("string"),
)
gravar_gold(dim_movies, "gold.dim_movies")
dim_movies = spark.table("gold.dim_movies")

dim_genres = (s_gen.select("genero").distinct()
              .select(sk("genero", prefixo="genre").alias("sk_genre_id"), F.col("genero").alias("nome_genero")))
gravar_gold(dim_genres, "gold.dim_genres")

dim_people = (s_pessoa.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
              .select("nome_entidade", "tipo_entidade").distinct()
              .select(sk("nome_entidade", "tipo_entidade", prefixo="person").alias("sk_person_id"),
                      F.col("nome_entidade").alias("nome_pessoa"),
                      F.col("tipo_entidade").alias("tipo_pessoa")))
gravar_gold(dim_people, "gold.dim_people")

dim_companies = (s_pessoa.filter(F.col("tipo_entidade") == "Produtora")
                 .select("nome_entidade").distinct()
                 .select(sk("nome_entidade", prefixo="company").alias("sk_company_id"),
                         F.col("nome_entidade").alias("nome_produtora")))
gravar_gold(dim_companies, "gold.dim_companies")

resumo_rev = (s_rev.groupBy("id_filme")
              .agg(F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),        
                   F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")))  
dim_reviews = (resumo_rev.join(dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")   
               .select(sk("id_filme", prefixo="review").alias("sk_review_id"), "sk_movie_id",
                       "qtd_avaliacoes_usuarios", "nota_media_usuarios"))
gravar_gold(dim_reviews, "gold.dim_reviews")

✔ gold.dim_movies                           97611 linhas
✔ gold.dim_genres                              20 linhas
✔ gold.dim_people                          418975 linhas
✔ gold.dim_companies                        45704 linhas
✔ gold.dim_reviews                          27226 linhas


### 1.2 Tabela Fato — `gold.fact_movies_performance`
* **Grão:** 1 linha por filme. Como `silver.tb_financeiro_filmes` e `silver.tb_metricas_engajamento` já são únicas por `id_filme`,
  os *joins* (todos `LEFT` a partir da `dim_movies`) **não multiplicam linhas**. Isso é verificado na seção de integridade.
* Tipos exatos exigidos: `DECIMAL(18,2)`, `DOUBLE`, `INT`.
* Widget `somente_lancados = sim` restringe a fato aos filmes com status **Lançado** (padrão `nao`: todos os filmes da dimensão).

In [0]:
base = dim_movies.select("sk_movie_id", "id_filme", "status_filme")
if SOMENTE_LANCADOS:
    base = base.filter(F.col("status_filme") == "Lançado")

fact = (
    base
    .join(s_fin.select("id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"),
          "id_filme", "left")
    .join(s_met.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"),
          "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"), F.col("receita_usd").cast("decimal(18,2)"), F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"), F.col("receita_brl").cast("decimal(18,2)"), F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"), F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"), F.col("qtd_votos_imdb").cast("int"),
    )
)
gravar_gold(fact, "gold.fact_movies_performance")

✔ gold.fact_movies_performance              97611 linhas


### 1.3 Tabelas-ponte (*bridge tables*)
Resolvem as relações **N:N** (filme × gênero / pessoa / produtora) sem duplicar a fato: a fato continua com 1 linha por filme
e as pontes são usadas apenas quando a análise precisa "abrir" gêneros, elenco ou produtoras.

In [0]:
mov = spark.table("gold.dim_movies").select("id_filme", "sk_movie_id")
dg  = spark.table("gold.dim_genres")
dp  = spark.table("gold.dim_people")
dc  = spark.table("gold.dim_companies")

bridge_genre = (s_gen.join(mov, "id_filme")
                .join(dg, s_gen["genero"] == dg["nome_genero"])
                .select("sk_movie_id", "sk_genre_id").distinct())
gravar_gold(bridge_genre, "gold.bridge_movie_genre")

bridge_person = (s_pessoa.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")).join(mov, "id_filme")
                 .join(dp, (F.col("nome_entidade") == dp["nome_pessoa"]) & (F.col("tipo_entidade") == dp["tipo_pessoa"]))
                 .select("sk_movie_id", "sk_person_id").distinct())
gravar_gold(bridge_person, "gold.bridge_movie_person")

bridge_company = (s_pessoa.filter(F.col("tipo_entidade") == "Produtora").join(mov, "id_filme")
                  .join(dc, F.col("nome_entidade") == dc["nome_produtora"])
                  .select("sk_movie_id", "sk_company_id").distinct())
gravar_gold(bridge_company, "gold.bridge_movie_company")

✔ gold.bridge_movie_genre                  139861 linhas
✔ gold.bridge_movie_person                 759177 linhas
✔ gold.bridge_movie_company                116606 linhas


### 1.4 Verificação de integridade do modelo
Garante: chaves primárias únicas, **fato sem duplicar grão**, nenhuma FK órfã nas pontes/fato/reviews.

In [0]:
def unico(tabela, chave):
    t = spark.table(tabela)
    total, distintos = t.agg(F.count(F.lit(1)), F.countDistinct(chave)).first()
    ok = total == distintos
    print(f"{'✔' if ok else '✘'} PK única  {tabela}.{chave}: {total} linhas / {distintos} chaves distintas")
    return ok

def sem_orfaos(tabela_fk, fk, tabela_pk, pk):
    filha, pai = spark.table(tabela_fk), spark.table(tabela_pk)
    orfaos = filha.join(pai, filha[fk] == pai[pk], "left_anti").count()
    print(f"{'✔' if orfaos == 0 else '✘'} FK       {tabela_fk}.{fk} -> {tabela_pk}.{pk}: {orfaos} órfãos")
    return orfaos == 0

resultados = [
    unico("gold.dim_movies", "sk_movie_id"), unico("gold.dim_genres", "sk_genre_id"),
    unico("gold.dim_people", "sk_person_id"), unico("gold.dim_companies", "sk_company_id"),
    unico("gold.dim_reviews", "sk_review_id"),
    unico("gold.fact_movies_performance", "sk_movie_id"),            
    sem_orfaos("gold.fact_movies_performance", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    sem_orfaos("gold.dim_reviews", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    sem_orfaos("gold.bridge_movie_genre", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    sem_orfaos("gold.bridge_movie_genre", "sk_genre_id", "gold.dim_genres", "sk_genre_id"),
    sem_orfaos("gold.bridge_movie_person", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    sem_orfaos("gold.bridge_movie_person", "sk_person_id", "gold.dim_people", "sk_person_id"),
    sem_orfaos("gold.bridge_movie_company", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    sem_orfaos("gold.bridge_movie_company", "sk_company_id", "gold.dim_companies", "sk_company_id"),
]
if not all(resultados):
    raise AssertionError("Falha na verificação de integridade do Star Schema (veja os ✘ acima).")
print("\nStar Schema íntegro ✔")

✔ PK única  gold.dim_movies.sk_movie_id: 97611 linhas / 97611 chaves distintas
✔ PK única  gold.dim_genres.sk_genre_id: 20 linhas / 20 chaves distintas
✔ PK única  gold.dim_people.sk_person_id: 418975 linhas / 418975 chaves distintas
✔ PK única  gold.dim_companies.sk_company_id: 45704 linhas / 45704 chaves distintas
✔ PK única  gold.dim_reviews.sk_review_id: 27226 linhas / 27226 chaves distintas
✔ PK única  gold.fact_movies_performance.sk_movie_id: 97611 linhas / 97611 chaves distintas
✔ FK       gold.fact_movies_performance.sk_movie_id -> gold.dim_movies.sk_movie_id: 0 órfãos
✔ FK       gold.dim_reviews.sk_movie_id -> gold.dim_movies.sk_movie_id: 0 órfãos
✔ FK       gold.bridge_movie_genre.sk_movie_id -> gold.dim_movies.sk_movie_id: 0 órfãos
✔ FK       gold.bridge_movie_genre.sk_genre_id -> gold.dim_genres.sk_genre_id: 0 órfãos
✔ FK       gold.bridge_movie_person.sk_movie_id -> gold.dim_movies.sk_movie_id: 0 órfãos
✔ FK       gold.bridge_movie_person.sk_person_id -> gold.dim_people.sk

## Entrega 2 — `gold.gold_genai_movies_context` (base do Vector Search / RAG)

Documento em **frase corrida** (não lista de campos) no template:

> *O filme [TÍTULO], lançado no ano de [ANO], faturou [RECEITA] e teve um custo de [ORÇAMENTO]. Estrelado por [ATORES PRINCIPAIS] e dirigido por [DIRETOR], o filme possui a seguinte sinopse: [OVERVIEW].*

### Os nulos
`concat()` devolve **NULL para a frase inteira** se *qualquer* argumento for NULL — um filme sem diretor ou sem sinopse sumiria do índice vetorial em silêncio.
Solução: **`coalesce(campo, fallback)`** em **cada** campo antes do `concat`.

| Campo | Chance de vir nulo? | Por quê | Fallback |
|---|---|---|---|
| título | baixa | título ausente/corrompido | `Título não informado` |
| ano | média | data de lançamento não convertível ou filme não lançado | `ano não informado` |
| receita | **alta** | 0/ausente na origem → `NULL` na Silver | `valor não informado` |
| orçamento | **alta** | idem | `valor não informado` |
| atores | média | filme sem elenco no cadastro (LEFT JOIN sem match) | `elenco não informado` |
| diretor | média | idem (e LEFT JOIN sem match) | `diretor não informado` |
| sinopse | média | `overview` vazio | `sinopse não disponível` |

Como há vários atores por filme, eles são **agregados numa única string** (`collect_list` + `array_join`), limitados aos **5 primeiros
pela ordem de créditos** (o que define "atores principais"), usando `bridge_movie_person` + `dim_people` filtrando `tipo_pessoa`.

In [0]:
dm  = spark.table("gold.dim_movies")
ft  = spark.table("gold.fact_movies_performance")
dp  = spark.table("gold.dim_people")
bp  = spark.table("gold.bridge_movie_person")

ordem = s_pessoa.select("id_filme", F.col("nome_entidade").alias("nome_pessoa"),
                        F.col("tipo_entidade").alias("tipo_pessoa"), "ordem_creditos")

pessoas_do_filme = (bp.join(dp, "sk_person_id")
                    .join(dm.select("sk_movie_id", "id_filme"), "sk_movie_id")
                    .join(ordem, ["id_filme", "nome_pessoa", "tipo_pessoa"], "left"))

def agrega_nomes(tipo: str, limite: int = None):
    """Agrega os nomes de um tipo de pessoa em UMA string por filme, respeitando a ordem de créditos."""
    p = pessoas_do_filme.filter(F.col("tipo_pessoa") == tipo)
    if limite:
        w = Window.partitionBy("sk_movie_id").orderBy(F.col("ordem_creditos").asc_nulls_last(), F.col("nome_pessoa"))
        p = p.withColumn("_pos", F.row_number().over(w)).filter(F.col("_pos") <= limite)
    return (p.groupBy("sk_movie_id")
             .agg(F.array_join(
                    F.transform(F.array_sort(F.collect_list(F.struct(F.coalesce("ordem_creditos", F.lit(999999)).alias("o"),
                                                                      F.col("nome_pessoa").alias("n")))),
                                lambda x: x["n"]), ", ").alias(f"lista_{tipo.lower()}")))

atores    = agrega_nomes("Ator", limite=5).withColumnRenamed("lista_ator", "atores_principais")
diretores = agrega_nomes("Diretor").withColumnRenamed("lista_diretor", "diretores")

def com_fallback(coluna, texto_fallback: str):
    """Trata NULL *e* string vazia/espaços: devolve o texto de fallback em vez de NULL."""
    return F.coalesce(F.when(F.trim(coluna) != "", F.trim(coluna)), F.lit(texto_fallback))

def moeda_usd(c):
    """Decimal -> 'US$ 1.234.567,00' (padrão brasileiro). NULL continua NULL (tratado pelo fallback)."""
    return F.concat(F.lit("US$ "), F.translate(F.format_number(c, 2), ",.", ".,"))

ctx = (
    dm.select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse")
    .join(ft.select("sk_movie_id", "receita_usd", "orcamento_usd"), "sk_movie_id", "left")
    .join(atores, "sk_movie_id", "left")
    .join(diretores, "sk_movie_id", "left")
)

titulo   = com_fallback(F.col("titulo"), "Título não informado")
ano      = com_fallback(F.col("ano_lancamento").cast("string"), "ano não informado")
receita  = com_fallback(moeda_usd(F.col("receita_usd")), "valor não informado")
orcament = com_fallback(moeda_usd(F.col("orcamento_usd")), "valor não informado")
elenco   = com_fallback(F.col("atores_principais"), "elenco não informado")
diretor  = com_fallback(F.col("diretores"), "diretor não informado")
sinopse  = com_fallback(F.regexp_replace(F.regexp_replace(F.col("sinopse"), r"\s+", " "), r"[\s\.]+$", ""), "sinopse não disponível")

gold_ctx = ctx.select(
    F.col("id_filme").alias("movie_id"),
    titulo.alias("title"),
    F.concat(
        F.lit("O filme "), titulo, F.lit(", lançado no ano de "), ano,
        F.lit(", faturou "), receita, F.lit(" e teve um custo de "), orcament,
        F.lit(". Estrelado por "), elenco, F.lit(" e dirigido por "), diretor,
        F.lit(", o filme possui a seguinte sinopse: "), sinopse, F.lit("."),
    ).alias("llm_context_document"),
)
gravar_gold(gold_ctx, "gold.gold_genai_movies_context")

try:
    spark.sql("ALTER TABLE gold.gold_genai_movies_context SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
except Exception as e:
    print("Aviso (CDF não habilitado):", str(e)[:120])

n_ctx, n_dim = spark.table("gold.gold_genai_movies_context").count(), spark.table("gold.dim_movies").count()
n_nulos = spark.table("gold.gold_genai_movies_context").filter(F.col("llm_context_document").isNull()).count()
print(f"Filmes na dim_movies: {n_dim} | documentos de contexto: {n_ctx} | documentos NULL: {n_nulos}")
assert n_ctx == n_dim and n_nulos == 0, "A tabela de contexto perdeu filmes ou tem documentos NULL"
display(spark.table("gold.gold_genai_movies_context").limit(10))

✔ gold.gold_genai_movies_context            97611 linhas
Filmes na dim_movies: 97611 | documentos de contexto: 97611 | documentos NULL: 0


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Izzy Jones, Erika Alexander, Aron Von Andrian, Steven Michael-O’hara, Tedroy Newell e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou valor não informado e teve um custo de valor não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Marcello Urgeghe, João Pedro Bénard, Isabel Abreu, Inês Pronto e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou valor não informado e teve um custo de valor não informado. Estrelado por Soulayman Rkiba, Gabrielle Cohen, Claire Chust, Maxime Pambet, Biyouna e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: sinopse não disponível."
1000030,58 Hours: The Baby Jessica Story,"O filme 58 Hours: The Baby Jessica Story, lançado no ano de 2021, faturou valor não informado e teve um custo de valor não informado. Estrelado por elenco não informado e dirigido por Mark Bone, o filme possui a seguinte sinopse: sinopse não disponível."
1000054,One Hundred Years and Hope,"O filme One Hundred Years and Hope, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por elenco não informado e dirigido por Takashi Nishihara, o filme possui a seguinte sinopse: In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
1000058,Homecoming,"O filme Homecoming, lançado no ano de 2023, faturou valor não informado e teve um custo de US$ 4.700.000,00. Estrelado por Aissatou Diallo Sagn

## 4. Desafio de Analytics — perguntas de negócio

> **Recorte temporal (perguntas 5 e 6):** a data limite superior é a **data de lançamento mais recente já realizada** na base
> (ignora datas futuras e filmes ainda não lançados). "Últimos 2 anos" = `(data_ref − 2 anos, data_ref]`; "últimos 5 anos" = `(data_ref − 5 anos, data_ref]`.

In [0]:
print("1) Receita total somada (em R$)")
display(spark.sql("""
    SELECT ROUND(SUM(receita_brl), 2) AS receita_total_brl,
           COUNT(receita_brl)         AS filmes_com_receita
    FROM gold.fact_movies_performance
"""))

1) Receita total somada (em R$)


receita_total_brl,filmes_com_receita
831204771718.04,3240


In [0]:
print("2) Top 5 filmes por popularidade")
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

2) Top 5 filmes por popularidade


titulo,popularidade
England 79,1.97900074645379E14
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0


In [0]:
print("3) Filmes por gênero")
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

3) Filmes por gênero


nome_genero,qtd_filmes
Drama,32127
Documentary,18927
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


In [0]:
print("4) Top 10 filmes por receita")
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, posicao_ranking
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    WHERE posicao_ranking <= 10
    ORDER BY posicao_ranking, titulo
"""))

4) Top 10 filmes por receita


titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,14439320000.00,1
Avatar: The Way of Water,2320250281.00,11965298674.09,2
AVENGERS: INFINITY WAR,2052415039.00,10584099114.62,3
spider-man: no way home,1921847111.00,9910773366.72,4
The Lion King,1663075401.00,8576313535.42,5
Top Gun: Maverick,1488732821.00,7677246284.61,6
Barbie,1428545028.00,7366863854.89,7
The Super Mario Bros. Movie,1355725263.00,6991339608.76,8
Black Panther,1349926083.00,6961433817.42,9
Star Wars: The Last Jedi,1332698830.00,6872594596.43,10


In [0]:
DATA_REF = spark.sql("""
    SELECT MAX(data_lancamento) FROM gold.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
""").first()[0]
if DATA_REF is None:
    raise RuntimeError("Não há filmes lançados com data válida na dim_movies.")
print(f"Data de referência (lançamento mais recente já realizado): {DATA_REF}")

print("5) Atores com mais participações (últimos 2 anos) — posicao = 1 é a resposta (pode haver empate)")
display(spark.sql(f"""
    SELECT p.nome_pessoa AS ator,
           COUNT(DISTINCT m.sk_movie_id) AS qtd_participacoes,
           DENSE_RANK() OVER (ORDER BY COUNT(DISTINCT m.sk_movie_id) DESC) AS posicao
    FROM gold.bridge_movie_person b
    JOIN gold.dim_people p ON p.sk_person_id = b.sk_person_id AND p.tipo_pessoa = 'Ator'
    JOIN gold.dim_movies m ON m.sk_movie_id = b.sk_movie_id
    WHERE m.status_filme = 'Lançado'
      AND m.data_lancamento >  add_months(DATE'{DATA_REF}', -24)
      AND m.data_lancamento <= DATE'{DATA_REF}'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC, ator
    LIMIT 10
"""))

Data de referência (lançamento mais recente já realizado): 2026-02-19
5) Atores com mais participações (últimos 2 anos) — posicao = 1 é a resposta (pode haver empate)


ator,qtd_participacoes,posicao
Kevin Hart,64,1
Nathalie Emmanuel,56,2
Ben Schwartz,55,3
John Cena,51,4
Paula Pell,45,5
Melissa Ponzio,31,6
Greg Kriek,28,7
Alon Mcklveen,27,8
Forrest Conoly,17,9
Cooper Tomlinson,12,10


In [0]:
print("6) Produtoras com maior lucro (últimos 5 anos) — a 1ª linha é a resposta; as demais mostram o top 10")
display(spark.sql(f"""
    SELECT c.nome_produtora,
           SUM(f.lucro_usd) AS lucro_total_usd,
           SUM(f.lucro_brl) AS lucro_total_brl,
           COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes_no_periodo
    FROM gold.bridge_movie_company bc
    JOIN gold.dim_companies c ON c.sk_company_id = bc.sk_company_id
    JOIN gold.dim_movies m ON m.sk_movie_id = bc.sk_movie_id
    JOIN gold.fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
    WHERE m.status_filme = 'Lançado'
      AND m.data_lancamento >  add_months(DATE'{DATA_REF}', -60)
      AND m.data_lancamento <= DATE'{DATA_REF}'
      AND f.lucro_usd IS NOT NULL
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 10
"""))

6) Produtoras com maior lucro (últimos 5 anos) — a 1ª linha é a resposta; as demais mostram o top 10


nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes_no_periodo
Universal Pictures,5772329679.00,29767326921.64,24
Marvel Studios,4953462823.00,25544512431.92,8
Columbia Pictures,3662050755.00,18884829538.45,12
Pascal Pictures,2701952454.00,13933698610.03,3
Illumination,2431353473.00,12538246724.91,3
Paramount,2239394101.00,11548331439.44,14
Kevin Feige Productions,2138205367.00,11026511257.08,4
20th Century Studios,2052737977.00,10585764473.60,6
Lightstorm Entertainment,1860250281.00,9593124674.09,1
Arad Productions,1359746601.00,7012077246.69,4
